In [1]:
!pip install kagglehub librosa soundfile xgboost scikit-learn joblib tqdm

In [2]:
import os
import glob
import numpy as np
import librosa
from tqdm import tqdm
import kagglehub

# 1. Download the latest version of the dataset
path = kagglehub.dataset_download("ahanaf101/bangla-deepfake-dataset")
print("Path to dataset files:", path)

100%|██████████| 4.59G/4.59G [03:35<00:00, 22.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ahanaf101/bangla-deepfake-dataset/versions/1


In [3]:
audio_files = []
labels = []

# Walk through the directory structure found via kagglehub
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.wav'):
            file_path = os.path.join(root, file)

            # Normalize path separators to safely check folder names
            normalized_path = os.path.normpath(file_path)
            path_components = normalized_path.split(os.sep)

            if 'Fake' in path_components:
                audio_files.append(file_path)
                labels.append(1)  # 1 represents Fake/Deepfake
            elif 'Real' in path_components:
                audio_files.append(file_path)
                labels.append(0)  # 0 represents Genuine Real Audio

print(f"Total audio files discovered: {len(audio_files)}")
print(f"Fake files: {sum(labels)} | Real files: {len(labels) - sum(labels)}")

Total audio files discovered: 25506
Fake files: 12757 | Real files: 12749


In [4]:
def extract_features(file_path):
    try:
        # Load audio (resample to 16kHz to standardize across recordings)
        audio, sample_rate = librosa.load(file_path, sr=16000, duration=5.0)

        # Extract features
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        chroma = librosa.feature.chroma_stft(y=audio, sr=sample_rate)
        mel = librosa.feature.melspectrogram(y=audio, sr=sample_rate)
        contrast = librosa.feature.spectral_contrast(y=audio, sr=sample_rate)

        # Aggregate features across time using mean and standard deviation
        feature_vector = np.hstack([
            np.mean(mfccs, axis=1), np.std(mfccs, axis=1),
            np.mean(chroma, axis=1),
            np.mean(mel, axis=1),
            np.mean(contrast, axis=1)
        ])
        return feature_vector
    except Exception as e:
        # If an audio file is corrupted, return None gracefully
        return None

In [5]:
X = []
y = []

print("Extracting features from audio files... This might take a few minutes.")
for idx, file_path in enumerate(tqdm(audio_files)):
    features = extract_features(file_path)
    if features is not None:
        X.append(features)
        y.append(labels[idx])

X = np.array(X)
y = np.array(y)

print(f"\nFeature Matrix Shape: {X.shape}")

Extracting features from audio files... This might take a few minutes.


 96%|█████████▌| 24504/25506 [16:20<00:31, 31.62it/s]/tmp/ipykernel_4283/735149965.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sample_rate = librosa.load(file_path, sr=16000, duration=5.0)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_4283/735149965.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sample_rate = librosa.load(file_path, sr=16000, duration=5.0)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
 96%|█████████▌| 24508/25506 [16:20<00:53, 18.70it/s]/tmp/ipykernel_4283/7351


Feature Matrix Shape: (25506, 227)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the dataset (Fixed the parameter name to 'test_size')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalize data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data successfully split and scaled!")

Data successfully split and scaled!


In [7]:
from xgboost import XGBClassifier

# Initialize model with robust hyper-parameters to avoid overfitting
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

print("Training model...")
model.fit(X_train_scaled, y_train)
print("Training complete!")

Training model...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:58:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training complete!


In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"=== Model Accuracy: {accuracy * 100:.2f}% ===")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Real Audio', 'Fake Deepfake']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

=== Model Accuracy: 99.57% ===

Classification Report:
               precision    recall  f1-score   support

   Real Audio       1.00      0.99      1.00      2550
Fake Deepfake       0.99      1.00      1.00      2552

     accuracy                           1.00      5102
    macro avg       1.00      1.00      1.00      5102
 weighted avg       1.00      1.00      1.00      5102

Confusion Matrix:
[[2535   15]
 [   7 2545]]


In [9]:
import joblib

# Save the trained model and the matching scaler
joblib.dump(model, 'bangla_deepfake_detector.joblib')
joblib.dump(scaler, 'audio_scaler.joblib')

print("Files saved successfully! Download 'bangla_deepfake_detector.joblib' and 'audio_scaler.joblib' from the sidebar files tab for your web app.")

Files saved successfully! Download 'bangla_deepfake_detector.joblib' and 'audio_scaler.joblib' from the sidebar files tab for your web app.


In [10]:
def predict_uploaded_audio(file_path, model_path='bangla_deepfake_detector.joblib', scaler_path='audio_scaler.joblib'):
    # Load assets
    trained_model = joblib.load(model_path)
    app_scaler = joblib.load(scaler_path)

    # Extract features from the single uploaded file
    feat = extract_features(file_path)
    if feat is None:
        return "Error: Could not read audio file."

    # Reshape, scale, and predict
    feat_scaled = app_scaler.transform(feat.reshape(1, -1))
    prediction = trained_model.predict(feat_scaled)[0]
    probability = trained_model.predict_proba(feat_scaled)[0]

    label_map = {0: "Real Audio", 1: "Fake Deepfake"}
    confidence = probability[prediction] * 100

    return f"Prediction: {label_map[prediction]} ({confidence:.2f}% confidence)"

# Test it locally inside Colab if you want:
# print(predict_uploaded_audio(audio_files[0]))

In [11]:
import numpy as np

print("=== DATASET SANITY CHECK ===")
print(f"Total audio files discovered: {len(audio_files)}")
print(f"Extracted feature matrix shape (X): {X.shape}")
print(f"Labels found: {len(y)} (Should match total audio files)")
print(f"Class Distribution -> Real (0): {list(y).count(0)} | Fake (1): {list(y).count(1)}")

if len(np.unique(y)) < 2:
    print("\n WARNING: Your model only found one class! Check folder names or path mapping.")
else:
    print("\n SUCCESS: Features and labels look structurally sound.")

=== DATASET SANITY CHECK ===
Total audio files discovered: 25506
Extracted feature matrix shape (X): (25506, 227)
Labels found: 25506 (Should match total audio files)
Class Distribution -> Real (0): 12749 | Fake (1): 12757

 SUCCESS: Features and labels look structurally sound.


In [12]:
import random

# Separate the discovered file list into real and fake for testing
real_test_profiles = [path for path, label in zip(audio_files, labels) if label == 0]
fake_test_profiles = [path for path, label in zip(audio_files, labels) if label == 1]

print("=== LIVE INFERENCE TEST RECONNAISSANCE ===\n")

# 1. Test a known Real File
if real_test_profiles:
    sample_real = random.choice(real_test_profiles)
    print(f" Testing Known REAL File: ...{os.path.sep.join(sample_real.split(os.sep)[-3:])}")
    # Call the inference function defined in Cell 10
    result_real = predict_uploaded_audio(sample_real)
    print(f" Result: {result_real}\n")
else:
    print(" No Real files available to test.")

# 2. Test a known Fake File
if fake_test_profiles:
    sample_fake = random.choice(fake_test_profiles)
    print(f" Testing Known FAKE File: ...{os.path.sep.join(sample_fake.split(os.sep)[-3:])}")
    # Call the inference function defined in Cell 10
    result_fake = predict_uploaded_audio(sample_fake)
    print(f" Result: {result_fake}\n")
else:
    print(" No Fake files available to test.")

=== LIVE INFERENCE TEST RECONNAISSANCE ===

 Testing Known REAL File: ...S13F05/Real/11.wav
 Result: Prediction: Real Audio (99.99% confidence)

 Testing Known FAKE File: ...SUST TTS Corpus/Fake/10018.wav
 Result: Prediction: Fake Deepfake (100.00% confidence)

